In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
# Configure GPU memory growth
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)

# Set mixed precision policy
tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [3]:
# Constants
BASE_DIR = "./content/Training"
BALANCED_DIR = "./content/Balanced"
CSV_PATH = "labels.csv"
RESIZE_WIDTH, RESIZE_HEIGHT = 224, 224
BATCH_SIZE = 32
EPOCHS = 25  # Increased epochs for better training
TARGET_SIZE_TUMOR = 155
TARGET_SIZE_STROKE = 950

In [4]:
# Data Augmentation
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=30,
    zoom_range=0.3,
    shear_range=0.2,
)

In [5]:
# Balance Dataset
def balance_class(src_class_dir, tgt_class_dir, target_size):
    os.makedirs(tgt_class_dir, exist_ok=True)
    files = os.listdir(src_class_dir)
    num_files = len(files)
    for file in files:
        src_path = os.path.join(src_class_dir, file)
        tgt_path = os.path.join(tgt_class_dir, file)
        if not os.path.exists(tgt_path):
            cv2.imwrite(tgt_path, cv2.imread(src_path))

    files = os.listdir(tgt_class_dir)
    if len(files) < target_size:
        for i in range(target_size - len(files)):
            file = files[i % len(files)]
            img = cv2.imread(os.path.join(tgt_class_dir, file))
            img = cv2.resize(img, (RESIZE_WIDTH, RESIZE_HEIGHT))
            img = np.expand_dims(img, axis=0)
            img = preprocess_input(img)
            augmented_img = next(datagen.flow(img, batch_size=1))[0].astype(np.uint8)
            new_file = f"aug_{i}_{file}"
            cv2.imwrite(os.path.join(tgt_class_dir, new_file), augmented_img)

# Apply Dataset Balancing
balance_class(os.path.join(BASE_DIR, "Tumor", "yes"), os.path.join(BALANCED_DIR, "Tumor", "yes"), TARGET_SIZE_TUMOR)
balance_class(os.path.join(BASE_DIR, "Tumor", "no"), os.path.join(BALANCED_DIR, "Tumor", "no"), TARGET_SIZE_TUMOR)
balance_class(os.path.join(BASE_DIR, "Stroke", "yes"), os.path.join(BALANCED_DIR, "Stroke", "yes"), TARGET_SIZE_STROKE)
balance_class(os.path.join(BASE_DIR, "Stroke", "no"), os.path.join(BALANCED_DIR, "Stroke", "no"), TARGET_SIZE_STROKE)

In [8]:
# Load Balanced Dataset
filenames, tumor_labels, stroke_labels = [], [], []
for condition in ["Tumor", "Stroke"]:
    for class_label in ["yes", "no"]:
        class_dir = os.path.join(BALANCED_DIR, condition, class_label)
        for file in os.listdir(class_dir):
            filenames.append(os.path.join(condition, class_label, file))
            tumor_labels.append(1 if condition == "Tumor" and class_label == "yes" else 0)
            stroke_labels.append(1 if condition == "Stroke" and class_label == "yes" else 0)

df = pd.DataFrame({"filename": filenames, "tumor": tumor_labels, "stroke": stroke_labels})
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_gen = ImageDataGenerator(preprocessing_function=preprocess_input)
val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Train Generator
train_data = train_gen.flow_from_dataframe(
    train_df,
    directory=BALANCED_DIR,
    x_col="filename",
    y_col=["tumor", "stroke"],  # Multilabel outputs
    target_size=(RESIZE_WIDTH, RESIZE_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode="raw"  # Returns raw arrays for multi-output
)

# Validation Generator
val_data = val_gen.flow_from_dataframe(
    val_df,
    directory=BALANCED_DIR,
    x_col="filename",
    y_col=["tumor", "stroke"],  # Multilabel outputs
    target_size=(RESIZE_WIDTH, RESIZE_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode="raw"  # Returns raw arrays for multi-output
)

# Model Definition
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(RESIZE_WIDTH, RESIZE_HEIGHT, 3))
for layer in base_model.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)

# Model Outputs
tumor_output = Dense(1, activation="sigmoid", name="tumor_output")(x)
stroke_output = Dense(1, activation="sigmoid", name="stroke_output")(x)

model = Model(inputs=base_model.input, outputs=[tumor_output, stroke_output])
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss=["binary_crossentropy", "binary_crossentropy"],  # One loss per output
    metrics=["accuracy"]
)

# Callbacks
callbacks = [
    ModelCheckpoint("model/best_model.keras", save_best_only=True, monitor="val_loss"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
]

# Train the Model
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

# Plot Training History
plt.figure(figsize=(12, 6))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.title("Training and Validation Loss")
plt.show()

# Save Final Model
model.save("model/final_model.keras")


Found 1785 validated image filenames.
Found 447 validated image filenames.
Epoch 1/25


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 2), output.shape=(None, 1)